# Импорт библиотек

In [1]:
# !pip install catboost pyarrow fastparquet

In [23]:
import re
import numpy as np
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from scipy.stats import mstats
import warnings

warnings.filterwarnings('ignore')

# Загрузка датасета

In [3]:
data = []
with open('cian_offers.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line))

data = pd.DataFrame(data)
data.drop(['id'], axis=1, inplace=True)

# Первичный просмотр данных

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29129 entries, 0 to 29128
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   price              29129 non-null  int64  
 1   description        29129 non-null  object 
 2   rooms              28215 non-null  float64
 3   area_total         29129 non-null  float64
 4   area_living        22194 non-null  float64
 5   area_kitchen       24217 non-null  float64
 6   floor              29129 non-null  int64  
 7   floors_total       29129 non-null  int64  
 8   construction_year  25632 non-null  float64
 9   address            29129 non-null  object 
 10  district           29129 non-null  object 
 11  metros             29082 non-null  object 
 12  flat_type          29129 non-null  object 
 13  ceiling_height     23486 non-null  float64
 14  renovation_type    17863 non-null  object 
 15  parking            20207 non-null  object 
 16  accident_rate      163

In [5]:
data.describe()

,price,rooms,area_total,area_living,area_kitchen,floor,floors_total,construction_year,ceiling_height,entrances
count,2.912900e+04,28215.000000,29129.000000,22194.000000,24217.000000,29129.000000,29129.000000,25632.000000,23486.000000,13585.000000
mean,4.546217e+07,2.127237,69.622515,38.638749,13.540695,10.941914,21.081603,2004.587274,3.014419,4.748473
std,6.305310e+07,1.184976,47.699342,28.953039,8.633225,10.779314,15.457210,31.432149,0.771079,24.391175
min,9.500000e+05,0.000000,0.000000,1.000000,0.600000,1.000000,1.000000,1785.000000,0.000000,1.000000
25%,1.802372e+07,1.000000,40.900000,20.000000,8.000000,4.000000,9.000000,1984.000000,2.700000,2.000000
50%,2.732812e+07,2.000000,60.000000,32.100000,10.700000,7.000000,17.000000,2022.000000,3.000000,4.000000
75%,4.850000e+07,3.000000,84.000000,47.000000,17.800000,15.000000,27.000000,2026.000000,3.100000,6.000000
max,2.833614e+09,5.000000,952.000000,842.000000,117.000000,84.000000,97.000000,2031.000000,46.000000,2024.000000


In [6]:
print(f"Дубликатов: {data.drop(['metros', 'photo_keys'], axis=1).duplicated().sum()}")

Дубликатов: 85


In [7]:
ind = data.drop(['metros', 'photo_keys'], axis=1).drop_duplicates().index
df = data.iloc[ind]

print(f"Дубликатов: {df.drop(['metros', 'photo_keys'], axis=1).duplicated().sum()}")

Дубликатов: 0


In [8]:
missing = df.isnull().sum() / len(df) * 100
print(missing[missing > 0].sort_values(ascending=False))

entrances            53.418951
accident_rate        43.833494
heating              43.833494
renovation_type      38.644815
parking              30.574301
area_living          23.788046
elevators            23.412753
bathroom             20.699628
ceiling_height       19.356838
house_type           18.795620
area_kitchen         16.839967
construction_year    11.985264
rooms                 3.140063
metros                0.161823
dtype: float64


# Feature Engineering

In [9]:
df_copy = df.copy()

## Feature Engineering целевой переменной

In [10]:
df = df[df['price'] < 1_000_000_000]
df['log_price'] = np.log1p(df['price'])

## Feature Engineering числовых признаков

In [11]:
df['rooms'] = df['rooms'].fillna(1)
df['area_living'] = df.groupby('rooms')['area_living'].transform(lambda x: x.fillna(x.median()))
df['area_kitchen'] = df.groupby('rooms')['area_kitchen'].transform(lambda x: x.fillna(x.median()))
df['construction_year'] = df['construction_year'].fillna(df['construction_year'].median())
df['ceiling_height'] = df['ceiling_height'].fillna(df['ceiling_height'].median())
df = df.drop('entrances', axis=1, errors='ignore')

In [12]:
df = df[df['rooms'] > 0]
df = df[df['rooms'] < 20]
df = df[df['area_total'] > 10]
df = df[df['area_total'] < 500]
df = df[df['area_living'] < df['area_total']]
df = df[df['area_kitchen'] < df['area_total']]
df = df[df['construction_year'] > 1800]
df = df[df['construction_year'] < 2030]
df = df[df['ceiling_height'] > 2]
df = df[df['ceiling_height'] < 10]
df = df[df['floor'] > 0]
df = df[df['floor'] <= df['floors_total']]

In [13]:
features_to_clip = ['area_total', 'area_living', 'area_kitchen', 'floor',
                    'floors_total', 'construction_year', 'ceiling_height']

for feature in features_to_clip:
    df[feature] = mstats.winsorize(df[feature], limits=[0.01, 0.01])

print(f"Осталось записей: {len(df)}")

Осталось записей: 25470


## Feature Engineering категориальных признаков

In [14]:
df = df.drop(['accident_rate', 'heating'], axis=1, errors='ignore')
df = df[~df['metros'].isna()]
df['renovation_type'] = df['renovation_type'].fillna('Не указано')
df['parking'] = df['parking'].fillna('Нет')
r = re.compile(r'\d+')
df['elevators'] = df['elevators'].fillna('Нет')
df['elevators_int'] = df['elevators'].apply(lambda x: np.array(r.findall(x)).astype(int).sum()).replace({0: 2})
df['bathroom'] = df['bathroom'].fillna('Нет')
df['bathroom_int'] = df['bathroom'].apply(lambda x: np.array(r.findall(x)).astype(int).sum()).replace({0: 1})
df['house_type'] = df['house_type'].fillna('Не указано')
df['metros_count'] = df['metros'].apply(lambda x: len(x))
df['metros_min_time'] = df['metros'].apply(lambda x: min(x[i]['time'] for i in range(len(x))))

## New Features

In [15]:
df['building_age'] = datetime.now().year - df['construction_year']
df['area_per_room'] = df['area_total'] / df['rooms']
df['floor_ratio'] = df['floor'] / df['floors_total']
df['kitchen_ratio'] = df['area_kitchen'] / df['area_total']
df['living_ratio'] = df['area_living'] / df['area_total']
df['has_parking'] = (df['parking'] != 'Нет').astype(int)
df['has_elevator'] = (df['elevators'] != 'Нет').astype(int)
df['is_new_building'] = (df['building_age'] <= 5).astype(int)

In [24]:
stop_list = df['district'].unique().tolist()[11:]
stop_list.remove('ЗелАО')
inds = []
for ind, row in df.iterrows():
    if row.district in stop_list:
        inds.append(ind)

df.drop(inds, inplace=True)
df['district'].replace({'НАО (Новомосковский)': 'НАО', 'ТАО (Троицкий)': 'ТАО'}, inplace=True)

In [38]:
df.to_parquet('clean_dataset.parquet')